https://colab.research.google.com/drive/1D_MJT60w3V6la-M1i4LQJp3HzJ1IFEji?usp=sharing

# **Naive Bayes Case Study on Titanic Dataset**

 apply the Naive Bayes classification algorithm to build a Titanic Survival
Prediction Model. This case study will help to understand probabilistic classification and how to implement Naive Bayes for real-world datasets.


**Import and explore the dataset (inspect columns, missing values, and data types).**

In [9]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Data Science Course/DATASETS/titanic.csv')

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## Dataset Overview

The Titanic dataset contains 891 passenger records and 12 features.

The target variable is **Survived**, where:
- 0 = Did Not Survive
- 1 = Survived

The dataset contains passenger demographic information, ticket details, and travel information that can be used to predict survival.

## Data Types Analysis

The dataset contains both numerical and categorical features.

Categorical features such as Sex and Embarked will require encoding before applying the Naive Bayes algorithm.

Missing values?

In [12]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [13]:
df = df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## Feature Selection

The following columns were removed:

- PassengerId
- Name
- Ticket
- Cabin

These features either contained excessive missing values or did not provide meaningful information for survival prediction.

In [14]:
df['Age'].describe()

,Age
count,714.000000
mean,29.699118
std,14.526497
min,0.420000
25%,20.125000
50%,28.000000
75%,38.000000
max,80.000000


In [16]:
df['Age'] = df['Age'].fillna(df['Age'].median())

In [17]:
df['Age'].isnull().sum()

np.int64(0)

In [18]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [19]:
df['Embarked'].value_counts()

,count
Embarked,
S,646
C,168
Q,77


In [20]:
df['Embarked'].isnull().sum()

np.int64(0)

In [21]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


####**Missing Value Treatment**

Missing values were handled using appropriate statistical techniques:

- Age: Missing values were replaced with the median age because median is robust to outliers.
- Embarked: Missing values were replaced with the mode because it is a categorical feature.
- Cabin: The feature was removed due to a large number of missing values.

**Encoding Categorical Variables :**

In [22]:
df.dtypes

,0
Survived,int64
Pclass,int64
Sex,object
Age,float64
SibSp,int64
Parch,int64
Fare,float64
Embarked,object


In [23]:
from sklearn.preprocessing import LabelEncoder

le_sex = LabelEncoder()

df['Sex'] = le_sex.fit_transform(df['Sex'])

In [24]:
le_embarked = LabelEncoder()

df['Embarked'] = le_embarked.fit_transform(df['Embarked'])

#### **Encoding Categorical Variables**

Machine learning algorithms require numerical input.

The categorical features `Sex` and `Embarked` were transformed into numerical values using Label Encoding.

In [25]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


**Split Features and Target:**

In [26]:
X = df.drop('Survived', axis=1)

y = df['Survived']

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [28]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (712, 7)
X_test : (179, 7)
y_train: (712,)
y_test : (179,)


#### Train-Test Split

The dataset was divided into training and testing sets using an 80:20 ratio.

The training set was used to train the model, while the testing set was used to evaluate its performance on unseen data.

## **Naive Bayes classifier**

In [29]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

GaussianNB()

#### Naive Bayes Model Training

Gaussian Naive Bayes was selected because the dataset contains continuous numerical features such as Age and Fare.

The model was trained using the training dataset to learn the probability distributions associated with passenger survival.

In [30]:
y_pred = nb.predict(X_test)

In [31]:
print(y_pred[:10])

[0 0 0 1 1 1 1 0 1 1]


In [32]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.776536312849162


In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[82 23]
 [17 57]]


#### Confusion Matrix Analysis

The confusion matrix provides a detailed breakdown of model predictions.

It shows how many passengers were correctly and incorrectly classified as survived or not survived.

In [34]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.78      0.80       105
           1       0.71      0.77      0.74        74

    accuracy                           0.78       179
   macro avg       0.77      0.78      0.77       179
weighted avg       0.78      0.78      0.78       179



## Model Evaluation Summary

The Gaussian Naive Bayes model was trained using the preprocessed Titanic dataset and evaluated on the test set.

### Performance Metrics

- Accuracy: **77.65%**
- Precision (Class 0 - Did Not Survive): **83%**
- Recall (Class 0 - Did Not Survive): **78%**
- Precision (Class 1 - Survived): **71%**
- Recall (Class 1 - Survived): **77%**

### Key Observations

- The model correctly classified most passengers with reasonable accuracy.
- It performed slightly better at predicting non-survivors than survivors.
- The confusion matrix showed 82 correct predictions for non-survivors and 57 correct predictions for survivors.
- Some misclassifications were observed, indicating scope for further improvement.

### Next Steps

Further model enhancement can be explored through feature engineering, feature selection, and model tuning. The current model serves as a baseline for comparison with future improvements.

This model serves as the baseline model for future improvements.

**Hyperparameter Tuning using GridSearchCV:**

In [35]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB

param_grid = {
    'var_smoothing': [1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7]
}

grid_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=GaussianNB(),
             param_grid={'var_smoothing': [1e-12, 1e-11, 1e-10, 1e-09, 1e-08,
                                           1e-07]},
             scoring='accuracy')

In [36]:
print("Best Parameters:", grid_search.best_params_)
print("Best Cross Validation Score:", grid_search.best_score_)

Best Parameters: {'var_smoothing': 1e-12}
Best Cross Validation Score: 0.7920713089727174


In [37]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

In [38]:
from sklearn.metrics import accuracy_score

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("Tuned Accuracy:", tuned_accuracy)

Tuned Accuracy: 0.776536312849162


Hyperparameter tuning was attempted using GridSearchCV, but no significant improvement was observed. Therefore, attention was shifted to feature engineering and feature selection

In [47]:
X3 = df.drop(['Survived', 'SibSp'], axis=1)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.2, random_state=42
)

from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7821229050279329


## Feature Selection Experiment

To investigate the impact of individual features, the `SibSp` feature was removed and the model was retrained.

### Results

| Model | Accuracy |
|---------|---------|
| Baseline Model | 77.65% |
| Without SibSp | 78.21% |

### Observation

A slight improvement in accuracy was observed after removing the `SibSp` feature. This indicates that the feature contributed limited predictive information and may have introduced redundancy with other family-related attributes.

Therefore, the model without `SibSp` was selected as the improved version.

## Findings and Insights

The Titanic dataset was analyzed and preprocessed by handling missing values, removing irrelevant features, and encoding categorical variables. A Gaussian Naive Bayes classifier was then applied to predict passenger survival.

The baseline model achieved an accuracy of **77.65%**. Further experimentation with feature selection showed that removing the **SibSp** feature improved the model accuracy to **78.21%**, indicating that this feature contributed limited predictive information for the Naive Bayes classifier.

Key insights from the analysis include:

- Gender was one of the strongest factors influencing survival, with female passengers having a higher likelihood of survival.
- Passenger class and fare were important indicators of survival probability.
- Age also played a role, as survival rates varied across different age groups.
- Feature selection improved model performance, demonstrating the importance of identifying and removing less useful attributes.
- The Gaussian Naive Bayes model provided a simple and effective approach for predicting passenger survival with satisfactory accuracy.

Overall, the case study demonstrated the complete machine learning workflow, including data exploration, preprocessing, model training, evaluation, and feature selection for performance improvement.